In [5]:
from pathlib import Path
import numpy as np
import pandas as pd

SEED_DIGIT = 3
STUDENT_ID_SUM = sum(int(ch) for ch in "267933")
RANDOM_SEED = int(f"{SEED_DIGIT}{STUDENT_ID_SUM}")

OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Locate secom.data or fallback CSV
data_path = None
for candidate in [
    Path("secom.data"),
    Path("data/secom.data"),
    Path("../data/secom.data"),
    Path("secom_data.csv"),
    Path("data/secom_data.csv"),
]:
    if candidate.exists():
        data_path = candidate
        break

if data_path is None:
    raise FileNotFoundError("Could not find secom.data or secom_data.csv in current/data folders.")

if data_path.suffix == ".csv":
    df_raw = pd.read_csv(data_path)
    feat_cols = [c for c in df_raw.columns if c.startswith("Feature_")]
    raw_measurements = df_raw[feat_cols].to_numpy(dtype=float)
else:
    raw_measurements = np.loadtxt(data_path, dtype=float)

print("Loaded from:", data_path)
print("raw_measurements shape:", raw_measurements.shape)
print("Random Seed:", RANDOM_SEED)

Loaded from: data\secom.data
raw_measurements shape: (1567, 590)
Random Seed: 330


In [6]:
# [APPENDIX 1] VECTORIZED MEASUREMENT QUALITY ANALYSIS

obs_counts = np.sum(~np.isnan(raw_measurements), axis=0)
usable_mask = obs_counts > 0
# Also exclude constant features among observed values
for idx in range(raw_measurements.shape[1]):
    col = raw_measurements[:, idx]
    valid = col[~np.isnan(col)]
    if valid.size <= 1 or np.nanmin(valid) == np.nanmax(valid):
        usable_mask[idx] = False

usable_indices = np.flatnonzero(usable_mask)

n_select = 5 + (SEED_DIGIT % 3)  # 5 + (3 % 3) = 5
rng_ex1 = np.random.default_rng(RANDOM_SEED)
selected_idx = np.sort(rng_ex1.choice(usable_indices, size=n_select, replace=False))
selected_names = [f"Feature_{i:03d}" for i in selected_idx]

sub = raw_measurements[:, selected_idx]  # shape: (N, 5)
n_rows = sub.shape[0]

missing_pct = np.mean(np.isnan(sub), axis=0) * 100.0
means = np.nanmean(sub, axis=0)
medians = np.nanmedian(sub, axis=0)
sds = np.nanstd(sub, axis=0, ddof=1)

abs_dev = np.abs(sub - medians)
mad = np.nanmedian(abs_dev, axis=0)

scale = 1.4826 * mad
robust_z = np.where(scale > 0, (sub - medians) / scale, 0.0)

flagged_mask = (~np.isnan(sub)) & (np.abs(robust_z) > 3.5)
flagged_counts = np.sum(flagged_mask, axis=0)
flagged_pct = (flagged_counts / n_rows) * 100.0

quality_df = pd.DataFrame({
    "Feature": selected_names,
    "Missing_Pct": np.round(missing_pct, 4),
    "Mean": np.round(means, 4),
    "Median": np.round(medians, 4),
    "Sample_SD": np.round(sds, 4),
    "MAD": np.round(mad, 4),
    "Flagged_Count": flagged_counts,
    "Flagged_Pct": np.round(flagged_pct, 4),
}).sort_values(by="Flagged_Pct", ascending=False).reset_index(drop=True)

quality_path = OUTPUT_DIR / "measurement_quality.csv"
quality_df.to_csv(quality_path, index=False)

highest_missing_pct = np.max(missing_pct)
top_flagged_feature = quality_df.iloc[0]["Feature"]
total_flagged = int(np.sum(flagged_mask))

loop_flagged = sum(
    int(np.sum((~np.isnan(sub[:, j])) & (np.abs(robust_z[:, j]) > 3.5)))
    for j in range(n_select)
)
vectorized_verified = total_flagged == loop_flagged

print(quality_df)
print("Selected Features:", selected_names)
print("Highest Missing Percentage:", f"{highest_missing_pct:.2f}%")
print("Feature with Highest Flagged Percentage:", top_flagged_feature)
print("Total Flagged Observations:", total_flagged)
print("Vectorized Results Verified:", vectorized_verified)
print("Saved:", quality_path.is_file())

       Feature  Missing_Pct      Mean    Median  Sample_SD      MAD  \
0  Feature_071       0.3829  104.3290  102.6043    31.6519  13.8556   
1  Feature_552      16.5922    0.2767    0.1979     0.2762   0.1030   
2  Feature_285       0.1914    2.1198    1.8752     0.9629   0.5313   
3  Feature_286       0.1914    4.2600    3.3600     9.7638   0.9200   
4  Feature_579      60.5616    0.0168    0.0148     0.0096   0.0046   

   Flagged_Count  Flagged_Pct  
0             84       5.3606  
1             57       3.6375  
2             33       2.1059  
3             21       1.3401  
4             21       1.3401  
Selected Features: ['Feature_071', 'Feature_285', 'Feature_286', 'Feature_552', 'Feature_579']
Highest Missing Percentage: 60.56%
Feature with Highest Flagged Percentage: Feature_071
Total Flagged Observations: 216
Vectorized Results Verified: True
Saved: True


In [10]:
# [APPENDIX 2] Mobile Device Usage Profiling

from pathlib import Path
import numpy as np
import pandas as pd

SEED_DIGIT = 3
ID_SUM = sum(int(d) for d in "267933")  # 30
RANDOM_SEED = 267933
OUTPUT_DIR = Path("output")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if "df" in globals():
    full_df = df.copy()
else:
    x_file = next(p for p in [Path("secom.data"), Path("data/secom.data"), Path("../data/secom.data")] if p.exists())
    y_file = next(p for p in [Path("secom_labels.data"), Path("data/secom_labels.data"), Path("../data/secom_labels.data")] if p.exists())
    X = np.loadtxt(x_file)
    y_df = pd.read_csv(y_file, sep=r"\s+", header=None)
    y = y_df.iloc[:, 0].astype(int).to_numpy()
    full_df = pd.DataFrame(X, columns=[f"Feature_{i:03d}" for i in range(X.shape[1])])
    full_df["Result"] = y
    full_df["Outcome"] = np.where(y == 1, "Fail", "Pass")

full_df["Record_ID"] = full_df.index

n_each = min(20 + ID_SUM, (full_df["Outcome"] == "Pass").sum(), (full_df["Outcome"] == "Fail").sum())
rng = np.random.default_rng(RANDOM_SEED)
bal_df = pd.concat([
    full_df[full_df["Outcome"] == "Pass"].sample(n=n_each, random_state=int(rng.integers(0, 10**9))),
    full_df[full_df["Outcome"] == "Fail"].sample(n=n_each, random_state=int(rng.integers(0, 10**9))),
]).sort_index()

feat_cols = [f"Feature_{i:03d}" for i in range(590)]
usable = [c for c in feat_cols if full_df[c].notna().sum() > 1 and full_df[c].min() != full_df[c].max()]
picked = list(rng.choice(usable, size=5, replace=False))

imp_df = full_df[picked].fillna(full_df[picked].median()).loc[bal_df.index].copy()
imp_df["Outcome"] = bal_df["Outcome"]

summary = []
for col in picked:
    g = imp_df.groupby("Outcome")[col].agg(["count", "mean", "median", "std"])
    n_p, n_f = int(g.loc["Pass", "count"]), int(g.loc["Fail", "count"])
    m_p, m_f = g.loc["Pass", "mean"], g.loc["Fail", "mean"]
    sd_p, sd_f = g.loc["Pass", "std"], g.loc["Fail", "std"]

    abs_diff = abs(m_f - m_p)
    pooled_sd = np.sqrt(((n_p - 1) * sd_p**2 + (n_f - 1) * sd_f**2) / (n_p + n_f - 2))
    smd = abs_diff / pooled_sd if pooled_sd > 0 else 0.0

    p_arr = imp_df.loc[imp_df["Outcome"] == "Pass", col].to_numpy()
    f_arr = imp_df.loc[imp_df["Outcome"] == "Fail", col].to_numpy()
    np_ok = (
        np.isclose(m_p, np.mean(p_arr)) and np.isclose(m_f, np.mean(f_arr))
        and np.isclose(sd_p, np.std(p_arr, ddof=1)) and np.isclose(sd_f, np.std(f_arr, ddof=1))
    )

    summary.append({
        "Feature": col,
        "Pass_Mean": round(m_p, 4),
        "Fail_Mean": round(m_f, 4),
        "Pass_Median": round(g.loc["Pass", "median"], 4),
        "Fail_Median": round(g.loc["Fail", "median"], 4),
        "Pass_SD": round(sd_p, 4),
        "Fail_SD": round(sd_f, 4),
        "Abs_Mean_Diff": round(abs_diff, 4),
        "Abs_SMD": round(smd, 4),
        "NumPy_Match": bool(np_ok),
    })

comp_df = pd.DataFrame(summary).sort_values("Abs_SMD", ascending=False).reset_index(drop=True)
bal_df[["Record_ID", "Outcome"] + picked].to_csv(OUTPUT_DIR / "balanced_sample.csv", index=False)
comp_df.to_csv(OUTPUT_DIR / "feature_comparison.csv", index=False)

print("Balanced Sample Size:", len(bal_df))
print("Selected Features:", picked)
print("Highest-Ranked Feature:", comp_df.loc[0, "Feature"])
print("Absolute Standardized Mean Difference:", f"{comp_df.loc[0, 'Abs_SMD']:.4f}")
print("NumPy-Pandas Statistics Match:", bool(comp_df["NumPy_Match"].all()))
print(comp_df) 

Balanced Sample Size: 100
Selected Features: [np.str_('Feature_552'), np.str_('Feature_251'), np.str_('Feature_558'), np.str_('Feature_130'), np.str_('Feature_283')]
Highest-Ranked Feature: Feature_130
Absolute Standardized Mean Difference: 0.6094
NumPy-Pandas Statistics Match: True
       Feature  Pass_Mean  Fail_Mean  Pass_Median  Fail_Median  Pass_SD  \
0  Feature_130     0.7265     0.7767       0.7438       0.7856   0.0965   
1  Feature_283     2.8481     2.6570       2.8644       2.4562   0.7455   
2  Feature_251     0.0015     0.0021       0.0010       0.0009   0.0024   
3  Feature_558     0.9846     0.9885       0.9720       0.9774   0.0521   
4  Feature_552     0.2656     0.2882       0.1916       0.1979   0.3570   

   Fail_SD  Abs_Mean_Diff  Abs_SMD  NumPy_Match  
0   0.0652         0.0502   0.6094         True  
1   1.0317         0.1911   0.2123         True  
2   0.0050         0.0006   0.1553         True  
3   0.0662         0.0039   0.0662         True  
4   0.3717     

In [11]:
# [APPENDIX 3] Statistical Robutness And Sensitivity Analysis

from pathlib import Path
import numpy as np
import pandas as pd

SEED_DIGIT = 3
RANDOM_SEED = 267933
N_BOOT = 200 + 10 * SEED_DIGIT  # 230 repetitions
OUT_DIR = Path("output")
OUT_DIR.mkdir(exist_ok=True)

# 1. Load data + labels safely
x_path = next(p for p in [Path("secom.data"), Path("data/secom.data"), Path("../data/secom.data")] if p.exists())
y_path = next(p for p in [Path("secom_labels.data"), Path("data/secom_labels.data"), Path("../data/secom_labels.data")] if p.exists())

X = np.loadtxt(x_path)
y = pd.read_csv(y_path, sep=r"\s+", header=None).iloc[:, 0].astype(int).to_numpy()

df_all = pd.DataFrame(X, columns=[f"Feature_{i:03d}" for i in range(X.shape[1])])
df_all["Outcome"] = np.where(y == 1, "Fail", "Pass")

pass_mask = (df_all["Outcome"] == "Pass").to_numpy()
fail_mask = (df_all["Outcome"] == "Fail").to_numpy()

# Select 5 usable features with >= 10 observed values in both Pass and Fail
feat_cols = [f"Feature_{i:03d}" for i in range(X.shape[1])]
usable = [
    c for c in feat_cols
    if df_all.loc[pass_mask, c].notna().sum() >= 10
    and df_all.loc[fail_mask, c].notna().sum() >= 10
    and df_all[c].min() != df_all[c].max()
]

rng = np.random.default_rng(RANDOM_SEED)
picked = list(rng.choice(usable, size=5, replace=False))

def run_strategy(df_sub, strategy_name, seed_val):
    r = np.random.default_rng(seed_val)
    out_rows = []
    for c in picked:
        s = df_sub[c]
        if strategy_name == "available_case":
            p_vals = s[pass_mask].dropna().to_numpy()
            f_vals = s[fail_mask].dropna().to_numpy()
        else:
            med = s.median()
            filled = s.fillna(med)
            p_vals = filled[pass_mask].to_numpy()
            f_vals = filled[fail_mask].to_numpy()

        # Fail minus Pass mean difference
        obs_diff = float(np.mean(f_vals) - np.mean(p_vals))

        # Stratified bootstrap resampling within each outcome group
        boot_diffs = np.empty(N_BOOT, dtype=float)
        for b in range(N_BOOT):
            p_boot = r.choice(p_vals, size=len(p_vals), replace=True)
            f_boot = r.choice(f_vals, size=len(f_vals), replace=True)
            boot_diffs[b] = np.mean(f_boot) - np.mean(p_boot)

        ci_low, ci_high = np.percentile(boot_diffs, [2.5, 97.5])
        out_rows.append({
            "Feature": c,
            "Strategy": strategy_name,
            "Obs_Mean_Diff_Fail_Minus_Pass": obs_diff,
            "Boot_Mean_Diff": float(np.mean(boot_diffs)),
            "CI_95_Low": float(ci_low),
            "CI_95_High": float(ci_high),
        })
    return pd.DataFrame(out_rows)

# Run twice with the same seed to check reproducibility
df_avail_1 = run_strategy(df_all, "available_case", RANDOM_SEED)
df_med_1 = run_strategy(df_all, "median_imputed", RANDOM_SEED + 1)

df_avail_2 = run_strategy(df_all, "available_case", RANDOM_SEED)
df_med_2 = run_strategy(df_all, "median_imputed", RANDOM_SEED + 1)

repro_ok = bool(
    np.allclose(df_avail_1["Boot_Mean_Diff"], df_avail_2["Boot_Mean_Diff"])
    and np.allclose(df_med_1["Boot_Mean_Diff"], df_med_2["Boot_Mean_Diff"])
)

# Compare strategies
m = df_avail_1.merge(df_med_1, on="Feature", suffixes=("_Avail", "_Med"))
m["Diff_Change"] = (m["Obs_Mean_Diff_Fail_Minus_Pass_Med"] - m["Obs_Mean_Diff_Fail_Minus_Pass_Avail"]).abs()
m["Rank_Avail"] = m["Obs_Mean_Diff_Fail_Minus_Pass_Avail"].abs().rank(ascending=False).astype(int)
m["Rank_Med"] = m["Obs_Mean_Diff_Fail_Minus_Pass_Med"].abs().rank(ascending=False).astype(int)
m["Sign_Changed"] = np.sign(m["Obs_Mean_Diff_Fail_Minus_Pass_Avail"]) != np.sign(m["Obs_Mean_Diff_Fail_Minus_Pass_Med"])
m["Rank_Changed"] = m["Rank_Avail"] != m["Rank_Med"]

largest_change = float(m["Diff_Change"].max())
sign_changed_cnt = int(m["Sign_Changed"].sum())
rank_changed_cnt = int(m["Rank_Changed"].sum())

m.round(4).to_csv(OUT_DIR / "robustness_comparison.csv", index=False)

report_text = (
    f"Selected Features: {picked}\n"
    f"Bootstrap Repetitions: {N_BOOT}\n"
    f"Largest Change in Mean Difference: {largest_change:.4f}\n"
    f"Features with Changed Difference Sign: {sign_changed_cnt}\n"
    f"Features with Changed Ranking: {rank_changed_cnt}\n"
    f"Reproducibility Verified: {repro_ok}\n"
)
(OUT_DIR / "sensitivity_report.txt").write_text(report_text, encoding="utf-8")

print("Selected Features:", picked)
print("Bootstrap Repetitions:", N_BOOT)
print("Largest Change in Mean Difference:", f"{largest_change:.4f}")
print("Features with Changed Difference Sign:", sign_changed_cnt)
print("Features with Changed Ranking:", rank_changed_cnt)
print("Reproducibility Verified:", repro_ok) 

Selected Features: [np.str_('Feature_285'), np.str_('Feature_554'), np.str_('Feature_138'), np.str_('Feature_056'), np.str_('Feature_252')]
Bootstrap Repetitions: 230
Largest Change in Mean Difference: 0.0269
Features with Changed Difference Sign: 0
Features with Changed Ranking: 0
Reproducibility Verified: True
